In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

SRC_PATH = Path("../assets/pink shuriken.png")  # 必要に応じてパス変更

# 1. 高速NumPy処理用に元画像を256x256にリサイズして保持（フル解像度は保存時に適用）
orig_full = Image.open(SRC_PATH).convert("RGBA")
orig_prev = orig_full.resize((256, 256), Image.Resampling.LANCZOS)


def recolor_rgba(pil_img: Image.Image, target_hue: float, sat_scale: float, val_scale: float, white_mix: float) -> Image.Image:
    arr = np.asarray(pil_img, dtype=np.float32) / 255.0
    rgb = arr[..., :3]
    alpha = arr[..., 3:4]

    hsv = mcolors.rgb_to_hsv(rgb)
    h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]

    # マゼンタ〜紫域 (0.70 <= h <= 0.99) かつ彩度があるピクセル（黒・白・ハイライトを保護）
    mask = (h >= 0.70) & (h <= 0.99) & (s > 0.15) & (alpha[..., 0] > 0)

    h_new = np.where(mask, target_hue, h)
    s_new = np.where(mask, np.clip(s * sat_scale, 0.0, 1.0), s)
    v_new = np.where(mask, np.clip(v * val_scale, 0.0, 1.0), v)

    rgb_out = mcolors.hsv_to_rgb(np.stack([h_new, s_new, v_new], axis=-1))
    if white_mix > 0:
        m3 = mask[..., None]
        rgb_out = np.where(m3, rgb_out * (1.0 - white_mix) + white_mix, rgb_out)

    rgba_out = np.concatenate([rgb_out, alpha], axis=-1)
    return Image.fromarray((np.clip(rgba_out, 0.0, 1.0) * 255).astype(np.uint8), mode="RGBA")



In [ ]:

# 2. スライダー＆ボタン定義
hue_slider = widgets.FloatSlider(value=0.545, min=0.45, max=0.68, step=0.005, description="Hue (青系):")
sat_slider = widgets.FloatSlider(value=0.65, min=0.0, max=1.5, step=0.05, description="Saturation:")
val_slider = widgets.FloatSlider(value=1.08, min=0.5, max=1.5, step=0.02, description="Brightness:")
white_slider = widgets.FloatSlider(value=0.25, min=0.0, max=0.8, step=0.05, description="Ice White:")

save_btn = widgets.Button(description="Save assets/favicon.png", button_style="primary", icon="save")
status_out = widgets.Output()


def on_save(_):
    with status_out:
        status_out.clear_output()
        final_img = recolor_rgba(
            orig_full,
            hue_slider.value,
            sat_slider.value,
            val_slider.value,
            white_slider.value,
        )
        Path("../assets").mkdir(exist_ok=True)
        final_img.resize((128, 128), Image.Resampling.LANCZOS).save("../assets/favicon.png")
        final_img.resize((64, 64), Image.Resampling.LANCZOS).save("../assets/favicon_64.png")
        print(
            f"✅ Saved assets/favicon.png (128x128) & favicon_64.png "
            f"[Hue={hue_slider.value:.3f}, Sat={sat_slider.value:.2f}, Val={val_slider.value:.2f}, White={white_slider.value:.2f}]"
        )


save_btn.on_click(on_save)


def update_preview(target_hue, sat_scale, val_scale, white_mix):
    prev = recolor_rgba(orig_prev, target_hue, sat_scale, val_scale, white_mix)
    prev_32 = prev.resize((32, 32), Image.Resampling.LANCZOS)

    fig, axes = plt.subplots(1, 3, figsize=(8.5, 3.0), dpi=120)
    # 1. ダークテーマ背景（ブラウザのダークタブ / アプリ背景想定）
    axes[0].set_facecolor("#0f172a")
    axes[0].imshow(prev)
    axes[0].set_title("Dark Tab (#0f172a)", fontsize=9)

    # 2. ライトテーマ背景（ブラウザの標準タブ想定）
    axes[1].set_facecolor("#f1f5f9")
    axes[1].imshow(prev)
    axes[1].set_title("Light Tab (#f1f5f9)", fontsize=9)

    # 3. 32x32 実寸相当プレビュー
    axes[2].set_facecolor("#1e293b")
    axes[2].imshow(prev_32)
    axes[2].set_title("32x32 Actual Size", fontsize=9)

    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.show()


out = widgets.interactive_output(
    update_preview,
    {
        "target_hue": hue_slider,
        "sat_scale": sat_slider,
        "val_scale": val_slider,
        "white_mix": white_slider,
    },
)

display(widgets.VBox([hue_slider, sat_slider, val_slider, white_slider, save_btn, status_out, out]))

In [6]:
from pathlib import Path
import numpy as np
from PIL import Image

img = Image.open("../assets/favicon.png").convert("RGBA")
arr = np.asarray(img, dtype=np.float32)
h, w, _ = arr.shape

# 画像の中心座標と各ピクセルの中心からの正規化距離 (0.0 〜 1.0)
cy, cx = (h - 1) / 2.0, (w - 1) / 2.0
yy, xx = np.ogrid[:h, :w]
dist = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2) / (min(h, w) / 2.0)

# 輝度 (0〜255)
lum = 0.299 * arr[..., 0] + 0.587 * arr[..., 1] + 0.114 * arr[..., 2]

# 1. 中央の黒いドーナツリング領域 (dist < 0.23 かつ 暗いピクセル) を白系へ反転
#    元の光沢グラデーション（暗いグレー〜黒）を活かして「255 - lum」で反転すると立体感が残ります
black_ring_mask = (dist < 0.23) & (lum < 140) & (arr[..., 3] > 0)

out = arr.copy()
inv_val = np.clip(255.0 - lum * 0.65, 215.0, 255.0)  # 真っ白〜わずかに陰影のある白
out[black_ring_mask, 0] = inv_val[black_ring_mask]
out[black_ring_mask, 1] = inv_val[black_ring_mask]
out[black_ring_mask, 2] = np.clip(inv_val[black_ring_mask] + 5.0, 0, 255)  # ほんのり青白く

# 2. 中央の白い穴 (dist < 0.105) を「背景透過 (alpha=0)」にしたい場合は以下を有効化（白のままなら不要）
center_hole_mask = dist < 0.102
out[center_hole_mask, 3] = 0.0

res = Image.fromarray(out.astype(np.uint8), mode="RGBA")
res.save("../assets/favicon_white_core.png")
res.resize((64, 64), Image.Resampling.LANCZOS).save("../assets/favicon_64.png")
print("Updated ../assets/favicon.png with white core ring!")

Updated ../assets/favicon.png with white core ring!


In [9]:
from pathlib import Path
import numpy as np
from PIL import Image

# ※ まだ黒リングが残っている状態の画像（または再生成した青白手裏剣）を入力にしてください
img = Image.open("../assets/favicon_black_core.png").convert("RGBA")
arr = np.asarray(img, dtype=np.float32)
h, w, _ = arr.shape

cy, cx = (h - 1) / 2.0, (w - 1) / 2.0
yy, xx = np.ogrid[:h, :w]
dist = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2) / (min(h, w) / 2.0)

lum = 0.299 * arr[..., 0] + 0.587 * arr[..., 1] + 0.114 * arr[..., 2]

# 中央の黒いリング領域 (dist < 0.23 かつ 暗いピクセル)
black_ring_mask = (dist < 0.23) & (lum < 140) & (arr[..., 3] > 0)

out = arr.copy()

# 中間色の灰色 (約 135〜175) にマッピングし、元の光沢の明暗差をそのまま残す
gray_val = np.clip(135.0 + lum * 0.45, 130.0, 180.0)
out[black_ring_mask, 0] = gray_val[black_ring_mask]
out[black_ring_mask, 1] = np.clip(gray_val[black_ring_mask] + 3.0, 0, 255)
out[black_ring_mask, 2] = np.clip(gray_val[black_ring_mask] + 10.0, 0, 255)  # わずかにクールグレー寄り

# 中央の穴を背景透過にする場合
center_hole_mask = dist < 0.102
out[center_hole_mask, 3] = 0.0

res = Image.fromarray(out.astype(np.uint8), mode="RGBA")
res.save("../assets/favicon_gray.png")
res.resize((64, 64), Image.Resampling.LANCZOS).save("../assets/favicon_64.png")
print("Updated ../assets/favicon.png with mid-gray core ring!")

Updated ../assets/favicon.png with mid-gray core ring!
